# Localization Simulation

Benchmark four trackers on one distance/angle localization scenario: the classical **CEKF** (centralized) and **DEKF** (diffusion) extended Kalman filters, the learned **DKN** (Distributed KalmanNet), and an optional **GNN-RNN** baseline.

Everything is driven by a trained experiment's `run.log`, and every figure is written to `<experiment>/notebook_plots/`.

**Segments**
1. Setup & imports
2. Experiment selection
3. Scenario setup & visualization
4. Single-trial comparison
5. Monte Carlo study (noise sweep)
6. Diffusion-coefficient network
7. dt-mismatch robustness
8. Node-count generalization
9. Inference-latency comparison

## 1. Setup & Imports

Load libraries, add the repository root to `sys.path`, and import the localization helpers - system/observation models, the classical filters, the DKN and GNN-RNN builders, checkpoint-path utilities, and plotting functions - used throughout the notebook.

In [ ]:
import importlib
from copy import deepcopy
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "utils").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

torch.set_default_dtype(torch.float32)

import utils.LocalizationScenario as localization_scenario_module
import utils.BaselineModels as baseline_models_module
importlib.reload(localization_scenario_module)
importlib.reload(baseline_models_module)

from experiments.graphkalmanprocess_hparams import LOCALIZATION_BASELINE
from utils.ClassicDistributedKalman import (
    centralized_extended_kalman_filter,
    diffusion_extended_kalman_filter_parallel_edge,
)
from utils.LocalizationScenario import (
    ConstantVelocityModel,
    DistanceAngleObservation,
    build_dkn_model,
    build_gnn_rnn_model,
    build_graph_data_for_dkn,
    create_distance_based_graph,
    dkn_model_path,
    dkn_run_name,
    experiment_time_steps,
    farthest_mismatch_dt_ratio,
    format_dt_ratio,
    generate_measurements,
    generate_node_positions,
    generate_trajectory,
    generate_trial_data,
    gnn_rnn_model_path,
    list_experiments,
    load_state_dict_checked,
    load_experiment_config,
    measurement_snr_db,
    nearest_nominal_dt_ratio,
    plot_graph,
    position_error_from_dekf,
    position_error_from_state_sequence,
    predict_gnn_rnn,
    plot_tracking_results,
    plot_trajectory_and_nodes,
    seed_everything,
    sync_torch_device,
)


## 2. Experiment Selection

List the trained `experiment_*` folders under `save_root` and print each one's key settings. The next segment loads the chosen experiment; its `run.log` alone drives the whole comparison.

In [ ]:
base_config = deepcopy(LOCALIZATION_BASELINE)
save_root = REPO_ROOT / base_config["save_root"]

experiments = list_experiments(save_root)
if not experiments:
    raise RuntimeError(f"No experiment_* folders found under {save_root}")

print("Available experiments:")
for i, exp in enumerate(experiments, start=1):
    tag = ""
    description = ""
    try:
        ecfg = load_experiment_config(exp)
    except FileNotFoundError:
        ecfg = None
    if ecfg is not None:
        exp_title = ecfg.get("title", "")
        title_str = f' - "{exp_title}"' if exp_title else ""
        tag = f"  (T={experiment_time_steps(ecfg)}, lr={ecfg.get('learning_rate','?')}){title_str}"
        description = ecfg.get("description", "")
    print(f"  [{i}] {exp.name}{tag}")
    if description:
        print(f"      {description}")


## 3. Scenario Setup & Visualization

Load the selected experiment's config and rebuild the scenario it was trained on: constant-velocity dynamics, distance/angle sensors, and the sensor graph. Available DKN / GNN-RNN checkpoints are discovered, the output folder `notebook_plots/` is created, and the sensor graph plus a sample trajectory are plotted so the geometry is clear before benchmarking.

In [ ]:
EXPERIMENT = 1

if not 1 <= EXPERIMENT <= len(experiments):
    raise ValueError(f"EXPERIMENT must be in [1, {len(experiments)}], got {EXPERIMENT}")
experiment_dir = experiments[EXPERIMENT - 1]
config_val = load_experiment_config(experiment_dir)
print(f"Loaded experiment: {experiment_dir.name}")
print(pd.Series(config_val))

seed_everything(config_val["seed"])

num_nodes = config_val["num_nodes"]
use_dt_mismatch = config_val.get("use_dt_mismatch", False)
dt_mismatch_values = config_val.get("dt_mismatch_values", [1.0])
default_dkn_dt_ratio = nearest_nominal_dt_ratio(dt_mismatch_values) if use_dt_mismatch else None

state_dimension = config_val["state_dimension"]
x_init = np.array(config_val["x0"], dtype=float).reshape(state_dimension, 1)
p0 = np.eye(state_dimension) * config_val["p0_scale"]

num_time_steps = experiment_time_steps(config_val)
num_trials = config_val["num_trials"]
process_noise_std = config_val["process_noise_std"]
time_delta = config_val["time_delta"]
measurement_noise_values = config_val["measurement_noise_values"]
node_positions = np.array(config_val["node_positions"], dtype=float)
f_system = ConstantVelocityModel(time_delta)
f_system_dkn = ConstantVelocityModel(time_delta * float(default_dkn_dt_ratio)) if use_dt_mismatch else f_system
h_system = DistanceAngleObservation(node_positions)
node_types = h_system.node_classification[:, 0].numpy().astype(int)

adjacency_matrix = create_distance_based_graph(
    node_positions,
    k_neighbors=config_val["k_neighbors"],
    seed=config_val["graph_seed"],
)
j_matrix = np.array(adjacency_matrix, dtype=float, copy=True)
np.fill_diagonal(j_matrix, 1.0)

dkn_dir = experiment_dir
gnn_rnn_dir = experiment_dir / "gnn-rnn"
available_noise_values = [
    r for r in measurement_noise_values
    if dkn_model_path(
        experiment_dir,
        r,
        use_dt_mismatch=use_dt_mismatch,
        default_dt_ratio=default_dkn_dt_ratio,
    ).exists()
]
missing_noise_values = [r for r in measurement_noise_values if r not in available_noise_values]
if missing_noise_values:
    print(f"Warning: missing DKN checkpoints for r={missing_noise_values} in {dkn_dir} - skipping.")
if not available_noise_values:
    raise RuntimeError(f"No DKN checkpoints found in {dkn_dir}")

gnn_rnn_available_noise_values = [r for r in available_noise_values if gnn_rnn_model_path(experiment_dir, r).exists()]
missing_gnn_rnn_noise_values = [r for r in available_noise_values if r not in gnn_rnn_available_noise_values]
if missing_gnn_rnn_noise_values:
    print(f"Warning: missing GNN-RNN checkpoints for r={missing_gnn_rnn_noise_values} in {gnn_rnn_dir} - omitting GNN-RNN for those rows.")
use_gnn_rnn = len(gnn_rnn_available_noise_values) > 0
measurement_noise_values = available_noise_values

notebook_plot_dir = experiment_dir / "notebook_plots"
notebook_plot_dir.mkdir(parents=True, exist_ok=True)

print(f"Train nodes:      {num_nodes}")
print(f"Time steps:       {num_time_steps}")
if use_dt_mismatch:
    print(f"DKN checkpoint dt ratio for non-mismatch sections: {default_dkn_dt_ratio}")
plot_graph(adjacency_matrix, node_positions, title="Localization Scenario Graph")
plot_trajectory_and_nodes(node_positions, node_types, generate_trajectory(f_system, x_init, num_time_steps, process_noise_std))


## 4. Single-Trial Comparison

Run a single trajectory and overlay the estimates from CEKF, DEKF, DKN, and (when a checkpoint exists) GNN-RNN on identical measurements - a quick qualitative look at tracking quality before the statistical sweeps.

In [ ]:
r_demo = 0.25 if 0.25 in measurement_noise_values else measurement_noise_values[0]
trajectory, measurements = generate_trial_data(f_system, h_system, x_init, num_time_steps, process_noise_std, r_demo)
r_array_demo = r_demo * np.ones(num_nodes)
graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)

x_hat_cekf = centralized_extended_kalman_filter(
    measurements=measurements,
    f_system=f_system,
    h_system=h_system,
    r_array=r_array_demo,
    q=process_noise_std,
    p0=p0,
    x0=x_init,
    time_steps=num_time_steps,
    node_num=num_nodes,
)
x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
    measurements=measurements,
    f_system=f_system,
    h_system=h_system,
    r_array=r_array_demo,
    q=process_noise_std,
    p0=p0,
    x0=x_init,
    j_matrix=j_matrix,
    time_steps=num_time_steps,
    node_num=num_nodes,
)

example_model_path = dkn_model_path(
    experiment_dir,
    r_demo,
    use_dt_mismatch=use_dt_mismatch,
    default_dt_ratio=default_dkn_dt_ratio,
)
kalman_process = build_dkn_model(config_val, f_system_dkn, r_demo, x_init)
kalman_process = load_state_dict_checked(kalman_process, example_model_path)
with torch.no_grad():
    x_hat_dkn = kalman_process(graph_data).squeeze().cpu().numpy().mean(axis=1)

x_hat_gnn_rnn = None
gnn_path_demo = gnn_rnn_model_path(experiment_dir, r_demo)
if gnn_path_demo.exists():
    gnn_rnn_process = build_gnn_rnn_model(config_val)
    gnn_rnn_process = load_state_dict_checked(gnn_rnn_process, gnn_path_demo)
    x_hat_gnn_rnn = predict_gnn_rnn(gnn_rnn_process, graph_data, h_system)

demo_tracking_plot_path = notebook_plot_dir / f"{dkn_run_name(r_demo, use_dt_mismatch=use_dt_mismatch, default_dt_ratio=default_dkn_dt_ratio)}_tracking_comparison.png"
plot_tracking_results(
    trajectory,
    x_hat_cekf,
    x_hat_dekf=x_hat_dekf,
    x_hat_dkn=x_hat_dkn,
    x_hat_gnn_rnn=x_hat_gnn_rnn,
    node_positions=node_positions,
    node_types=node_types,
    save_path=demo_tracking_plot_path,
)


## 5. Monte Carlo Study (Noise Sweep)

For each measurement-noise level, average the position error over many random trials. CEKF, DEKF, and DKN are each evaluated with matched and mismatched `dt` dynamics, while GNN-RNN (no explicit dynamics) is evaluated once. Following the linear/non-linear notebooks, results are plotted with **position error in dB** against the inverse noise variance **$1/r^2$ (dB)** and against the empirical SNR (dB).

In [ ]:
if not use_dt_mismatch:
    raise RuntimeError("Monte Carlo mismatch comparison requires an experiment with use_dt_mismatch=true.")

MONTE_CARLO_NO_MISMATCH_RATIO = nearest_nominal_dt_ratio(dt_mismatch_values)
MONTE_CARLO_MISMATCH_RATIO = farthest_mismatch_dt_ratio(dt_mismatch_values, MONTE_CARLO_NO_MISMATCH_RATIO)
monte_carlo_dt_ratios = [MONTE_CARLO_NO_MISMATCH_RATIO, MONTE_CARLO_MISMATCH_RATIO]

monte_carlo_labels = {
    model_name: {ratio: f"{model_name} dt x{format_dt_ratio(ratio)}" for ratio in monte_carlo_dt_ratios}
    for model_name in ["CEKF", "DEKF", "DKN"]
}
monte_carlo_series = [
    monte_carlo_labels["CEKF"][MONTE_CARLO_NO_MISMATCH_RATIO],
    monte_carlo_labels["CEKF"][MONTE_CARLO_MISMATCH_RATIO],
    monte_carlo_labels["DEKF"][MONTE_CARLO_NO_MISMATCH_RATIO],
    monte_carlo_labels["DEKF"][MONTE_CARLO_MISMATCH_RATIO],
    monte_carlo_labels["DKN"][MONTE_CARLO_NO_MISMATCH_RATIO],
    monte_carlo_labels["DKN"][MONTE_CARLO_MISMATCH_RATIO],
    "GNN-RNN",
]

print(f"Monte Carlo dt ratios: no mismatch={format_dt_ratio(MONTE_CARLO_NO_MISMATCH_RATIO)}, mismatch={format_dt_ratio(MONTE_CARLO_MISMATCH_RATIO)}")

avg_snr_db = []
avg_errors = {label: [] for label in monte_carlo_series}

for r_noise in measurement_noise_values:
    r_array = r_noise * np.ones(num_nodes)

    dkn_processes = {}
    for dt_ratio in monte_carlo_dt_ratios:
        model_path = dkn_model_path(experiment_dir, r_noise, use_dt_mismatch=use_dt_mismatch, dt_ratio=dt_ratio)
        if not model_path.exists():
            raise FileNotFoundError(f"Missing DKN checkpoint for Monte Carlo comparison: {model_path}")
        f_model_ratio = ConstantVelocityModel(time_delta * float(dt_ratio))
        kalman_process = build_dkn_model(config_val, f_model_ratio, r_noise, x_init)
        dkn_processes[dt_ratio] = load_state_dict_checked(kalman_process, model_path)

    gnn_rnn_process = None
    gnn_path = gnn_rnn_model_path(experiment_dir, r_noise)
    if gnn_path.exists():
        gnn_rnn_process = build_gnn_rnn_model(config_val)
        gnn_rnn_process = load_state_dict_checked(gnn_rnn_process, gnn_path)
    else:
        print(f"No GNN-RNN checkpoint found for r={r_noise}; GNN-RNN result will be NaN.")

    trial_snr_db = []
    trial_errors = {label: [] for label in monte_carlo_series}

    for _ in tqdm(range(num_trials), desc=f"r={r_noise}"):
        trajectory, measurements = generate_trial_data(f_system, h_system, x_init, num_time_steps, process_noise_std, r_noise)
        trial_snr_db.append(measurement_snr_db(h_system, trajectory, measurements))
        graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)

        for dt_ratio in monte_carlo_dt_ratios:
            f_model_ratio = ConstantVelocityModel(time_delta * float(dt_ratio))
            x_hat_cekf = centralized_extended_kalman_filter(
                measurements=measurements,
                f_system=f_model_ratio,
                h_system=h_system,
                r_array=r_array,
                q=process_noise_std,
                p0=p0,
                x0=x_init,
                time_steps=num_time_steps,
                node_num=num_nodes,
            )
            x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
                measurements=measurements,
                f_system=f_model_ratio,
                h_system=h_system,
                r_array=r_array,
                q=process_noise_std,
                p0=p0,
                x0=x_init,
                j_matrix=j_matrix,
                time_steps=num_time_steps,
                node_num=num_nodes,
            )
            with torch.no_grad():
                x_hat_dkn = dkn_processes[dt_ratio](graph_data).squeeze().cpu().numpy().mean(axis=1)

            trial_errors[monte_carlo_labels["CEKF"][dt_ratio]].append(position_error_from_state_sequence(trajectory, x_hat_cekf[:, :, 0]))
            trial_errors[monte_carlo_labels["DEKF"][dt_ratio]].append(position_error_from_dekf(trajectory, x_hat_dekf))
            trial_errors[monte_carlo_labels["DKN"][dt_ratio]].append(position_error_from_state_sequence(trajectory, x_hat_dkn))

        if gnn_rnn_process is not None:
            x_hat_gnn_rnn = predict_gnn_rnn(gnn_rnn_process, graph_data, h_system)
            trial_errors["GNN-RNN"].append(position_error_from_state_sequence(trajectory, x_hat_gnn_rnn))

    avg_snr_db.append(np.mean(trial_snr_db))
    for label in monte_carlo_series:
        avg_errors[label].append(np.mean(trial_errors[label]) if trial_errors[label] else np.nan)

results_df = pd.DataFrame(
    {
        "measurement_noise": measurement_noise_values,
        "snr_db": avg_snr_db,
        **avg_errors,
    }
)
results_df


In [ ]:
summary_plot_path = notebook_plot_dir / f"localization_simulation_summary_mismatch_trials={num_trials}_T={num_time_steps}.png"
monte_carlo_plot_styles = {
    monte_carlo_labels["CEKF"][MONTE_CARLO_NO_MISMATCH_RATIO]: "o-",
    monte_carlo_labels["CEKF"][MONTE_CARLO_MISMATCH_RATIO]: "o--",
    monte_carlo_labels["DEKF"][MONTE_CARLO_NO_MISMATCH_RATIO]: "s-",
    monte_carlo_labels["DEKF"][MONTE_CARLO_MISMATCH_RATIO]: "s--",
    monte_carlo_labels["DKN"][MONTE_CARLO_NO_MISMATCH_RATIO]: "^-",
    monte_carlo_labels["DKN"][MONTE_CARLO_MISMATCH_RATIO]: "^--",
    "GNN-RNN": "d:",
}


inv_r2_db = 10 * np.log10(1.0 / results_df["measurement_noise"] ** 2)
plt.figure(figsize=(11, 7))
plotted_labels = []
for label, style in monte_carlo_plot_styles.items():
    if label in results_df.columns and results_df[label].notna().any():
        plt.plot(inv_r2_db, 10 * np.log10(results_df[label]), style, linewidth=2, markersize=7, label=label)
        plotted_labels.append(label)
print(f"Plotted {len(plotted_labels)} Monte Carlo error-vs-(1/r^2) lines: {plotted_labels}")
plt.xlabel(r"$1/r^2$ [dB]")
plt.ylabel("Position Error [dB]")
plt.title(f"Localization Simulation ({num_trials} trials, q={process_noise_std}, T={num_time_steps}, learn_edge_kalman={config_val['learn_edge_kalman']})")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.savefig(summary_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved summary plot: {summary_plot_path}")
plt.show()

snr_summary_plot_path = notebook_plot_dir / f"localization_simulation_snr_vs_error_mismatch_trials={num_trials}_T={num_time_steps}.png"
snr_results_df = results_df.sort_values("snr_db")
plt.figure(figsize=(11, 7))
plotted_snr_labels = []
for label, style in monte_carlo_plot_styles.items():
    if label in snr_results_df.columns and snr_results_df[label].notna().any():
        plt.plot(snr_results_df["snr_db"], 10 * np.log10(snr_results_df[label]), style, linewidth=2, markersize=7, label=label)
        plotted_snr_labels.append(label)
print(f"Plotted {len(plotted_snr_labels)} Monte Carlo SNR-vs-error lines: {plotted_snr_labels}")
plt.xlabel("Empirical Measurement SNR (dB)")
plt.ylabel("Position Error [dB]")
plt.title(f"SNR vs Error ({num_trials} trials, q={process_noise_std}, T={num_time_steps})")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2)
plt.savefig(snr_summary_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved SNR summary plot: {snr_summary_plot_path}")
plt.show()





## 6. Diffusion-Coefficient Network

*Does the network of diffusion coefficients the DEKF uses to combine neighbors actually change the results?* Today `j_matrix` is a trivial binary (uniform) network. Here it is rebuilt on the **same** sensor graph under several combination-weight schemes - `current (binary)`, `uniform (normalized)`, Metropolis-Hastings, and inverse-distance - and the DEKF noise sweep is rerun for each.

The DEKF also folds the coefficient row-sum into its process-noise term, so the three non-trivial schemes are scaled to a common row-sum (identical process-noise scaling); differences among them therefore isolate the *shape* of the coefficient network, while `current (binary)` shows today's behavior and CEKF is a network-free reference. The printed spread answers the "if at all" directly.

In [ ]:
DIFFUSION_NUM_TRIALS = num_trials
diff_f_system = ConstantVelocityModel(time_delta)   # true dynamics, no dt mismatch


def _sinkhorn_symmetric(weight, iters=200, eps=1e-12):
    """Scale a symmetric non-negative matrix to (approx.) doubly stochastic, preserving zeros."""
    w = np.array(weight, dtype=float, copy=True)
    for _ in range(iters):
        w = w / (w.sum(axis=1, keepdims=True) + eps)
        w = w / (w.sum(axis=0, keepdims=True) + eps)
    return 0.5 * (w + w.T)


def build_diffusion_coefficients(adjacency, node_positions, scheme):
    """Symmetric combination-weight matrix ('network of diffusion coefficients') on the
    fixed graph support (neighbors + self-loops)."""
    support = np.array(adjacency, dtype=float, copy=True)
    np.fill_diagonal(support, 0.0)
    neighbor_mask = support > 0
    degree = neighbor_mask.sum(axis=1)
    n = support.shape[0]
    rows, cols = np.where(neighbor_mask)

    if scheme == "current (binary)":
        return neighbor_mask.astype(float) + np.eye(n)
    if scheme == "uniform (normalized)":
        return _sinkhorn_symmetric(neighbor_mask.astype(float) + np.eye(n))
    if scheme == "metropolis":
        w = np.zeros((n, n))
        for i, j in zip(rows, cols):
            w[i, j] = 1.0 / (1.0 + max(degree[i], degree[j]))
        np.fill_diagonal(w, 1.0 - w.sum(axis=1))
        return w
    if scheme == "inverse-distance":
        w = np.zeros((n, n))
        for i, j in zip(rows, cols):
            w[i, j] = 1.0 / (1.0 + np.linalg.norm(node_positions[i] - node_positions[j]))
        np.fill_diagonal(w, 1.0)
        return _sinkhorn_symmetric(w)
    raise ValueError(f"Unknown scheme: {scheme}")


# The DEKF scales its process noise by the coefficient row-sum, so put every non-trivial
# scheme on a common row-sum => identical process-noise term (isolates the *shape* of the network).
_support = np.array(adjacency_matrix, dtype=float, copy=True)
np.fill_diagonal(_support, 0.0)
diffusion_target_scale = float(((_support > 0).sum(axis=1) + 1.0).mean())

diffusion_schemes = ["current (binary)", "uniform (normalized)", "metropolis", "inverse-distance"]
diffusion_j_matrices = {}
for scheme in diffusion_schemes:
    w = build_diffusion_coefficients(adjacency_matrix, node_positions, scheme)
    if scheme != "current (binary)":
        w = w * diffusion_target_scale
    diffusion_j_matrices[scheme] = w

diffusion_errors = {scheme: [] for scheme in diffusion_schemes}
diffusion_errors_cekf = []

for r_noise in measurement_noise_values:
    r_array = r_noise * np.ones(num_nodes)
    trials = [
        generate_trial_data(diff_f_system, h_system, x_init, num_time_steps, process_noise_std, r_noise)
        for _ in range(DIFFUSION_NUM_TRIALS)
    ]
    cekf_errs = []
    scheme_errs = {scheme: [] for scheme in diffusion_schemes}
    for trajectory, measurements in tqdm(trials, desc=f"diffusion r={r_noise}"):
        x_hat_cekf = centralized_extended_kalman_filter(
            measurements=measurements, f_system=diff_f_system, h_system=h_system,
            r_array=r_array, q=process_noise_std, p0=p0, x0=x_init,
            time_steps=num_time_steps, node_num=num_nodes,
        )
        cekf_errs.append(position_error_from_state_sequence(trajectory, x_hat_cekf[:, :, 0]))
        for scheme in diffusion_schemes:
            x_hat_dekf = diffusion_extended_kalman_filter_parallel_edge(
                measurements=measurements, f_system=diff_f_system, h_system=h_system,
                r_array=r_array, q=process_noise_std, p0=p0, x0=x_init,
                j_matrix=diffusion_j_matrices[scheme],
                time_steps=num_time_steps, node_num=num_nodes,
            )
            scheme_errs[scheme].append(position_error_from_dekf(trajectory, x_hat_dekf))
    diffusion_errors_cekf.append(float(np.mean(cekf_errs)))
    for scheme in diffusion_schemes:
        diffusion_errors[scheme].append(float(np.mean(scheme_errs[scheme])))

diffusion_results_df = pd.DataFrame({
    "measurement_noise": measurement_noise_values,
    "CEKF (reference)": diffusion_errors_cekf,
    **{f"DEKF: {scheme}": diffusion_errors[scheme] for scheme in diffusion_schemes},
})

diffusion_plot_path = notebook_plot_dir / f"diffusion_coefficient_network_trials={DIFFUSION_NUM_TRIALS}_T={num_time_steps}.png"
diffusion_inv_r2_db = 10 * np.log10(1.0 / np.array(measurement_noise_values, dtype=float) ** 2)
plt.figure(figsize=(11, 7))
plt.plot(diffusion_inv_r2_db, 10 * np.log10(diffusion_errors_cekf), "kx:", linewidth=1.5, markersize=8, label="CEKF (no network, reference)")
diffusion_styles = {
    "current (binary)": "o-",
    "uniform (normalized)": "s--",
    "metropolis": "^-.",
    "inverse-distance": "d:",
}
for scheme in diffusion_schemes:
    plt.plot(diffusion_inv_r2_db, 10 * np.log10(diffusion_errors[scheme]), diffusion_styles[scheme],
             linewidth=2, markersize=7, label=f"DEKF: {scheme}")
plt.xlabel(r"$1/r^2$ [dB]")
plt.ylabel("Position Error [dB]")
plt.title(f"Diffusion-Coefficient Network Comparison ({DIFFUSION_NUM_TRIALS} trials, q={process_noise_std}, T={num_time_steps})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(diffusion_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved: {diffusion_plot_path}")
plt.show()

matched_family = ["uniform (normalized)", "metropolis", "inverse-distance"]
matched_stack = np.array([diffusion_errors[s] for s in matched_family])
all_stack = np.array([diffusion_errors[s] for s in diffusion_schemes])
shape_spread = float(np.max(np.abs(matched_stack - matched_stack.mean(0)) / matched_stack.mean(0)))
overall_spread = float(np.max(np.abs(all_stack - all_stack.mean(0)) / all_stack.mean(0)))
verdict = "negligible" if shape_spread < 0.03 else ("small" if shape_spread < 0.10 else "significant")
print(f"\nMax relative spread across matched-Q coefficient networks (shape effect): {shape_spread:.1%}")
print(f"Max relative spread across all schemes (incl. current binary):           {overall_spread:.1%}")
print(f"=> Influence of the diffusion-coefficient network on DEKF error: {verdict}")

## 7. dt-Mismatch Robustness

Fix the measurement noise and sweep the `dt` mismatch multiplier. Data always uses the true `time_delta`; each filter instead runs with `time_delta x ratio`, and the DKN checkpoint trained at each ratio is loaded (the GNN-RNN checkpoint, if present, is reused for every ratio).

In [ ]:
if not config_val.get("use_dt_mismatch", False):
    raise RuntimeError("Set use_dt_mismatch=true in the experiment config to run this section.")

R_MISMATCH_EVAL = 1.0
mismatch_experiment_dir = experiment_dir
mismatch_config = config_val
dt_mismatch_values = mismatch_config.get("dt_mismatch_values", [1.0])

print(f"Experiment:         {mismatch_experiment_dir.name}")
print(f"dt_mismatch_values: {dt_mismatch_values}")
print(f"Evaluating at r={R_MISMATCH_EVAL}")

In [ ]:
mismatch_errors_cekf = []
mismatch_errors_dekf = []
mismatch_errors_dkn = []
mismatch_errors_gnn_rnn = []

r_array_mismatch = R_MISMATCH_EVAL * np.ones(num_nodes)
f_data = ConstantVelocityModel(time_delta)
mismatch_gnn_rnn_process = None
mismatch_gnn_rnn_path = gnn_rnn_model_path(mismatch_experiment_dir, R_MISMATCH_EVAL)
if mismatch_gnn_rnn_path.exists():
    mismatch_gnn_rnn_process = build_gnn_rnn_model(mismatch_config)
    mismatch_gnn_rnn_process = load_state_dict_checked(mismatch_gnn_rnn_process, mismatch_gnn_rnn_path)
else:
    print(f"No GNN-RNN checkpoint found at {mismatch_gnn_rnn_path}; omitting GNN-RNN from mismatch sweep.")

for ratio in dt_mismatch_values:
    f_model_r = ConstantVelocityModel(time_delta * ratio)

    model_path = dkn_model_path(
        mismatch_experiment_dir,
        R_MISMATCH_EVAL,
        use_dt_mismatch=mismatch_config.get("use_dt_mismatch", False),
        dt_ratio=ratio,
    )
    kalman_process_m = build_dkn_model(mismatch_config, f_model_r, R_MISMATCH_EVAL, x_init)
    kalman_process_m = load_state_dict_checked(kalman_process_m, model_path)

    trial_errors_cekf_m = []
    trial_errors_dekf_m = []
    trial_errors_dkn_m = []
    trial_errors_gnn_rnn_m = []

    for _ in tqdm(range(num_trials), desc=f"ratio={ratio}"):
        trajectory = generate_trajectory(f_data, x_init, num_time_steps, process_noise_std)
        measurements = generate_measurements(h_system, trajectory, R_MISMATCH_EVAL)

        x_hat_cekf_m = centralized_extended_kalman_filter(
            measurements=measurements,
            f_system=f_model_r,
            h_system=h_system,
            r_array=r_array_mismatch,
            q=process_noise_std,
            p0=p0,
            x0=x_init,
            time_steps=num_time_steps,
            node_num=num_nodes,
        )
        x_hat_dekf_m = diffusion_extended_kalman_filter_parallel_edge(
            measurements=measurements,
            f_system=f_model_r,
            h_system=h_system,
            r_array=r_array_mismatch,
            q=process_noise_std,
            p0=p0,
            x0=x_init,
            j_matrix=j_matrix,
            time_steps=num_time_steps,
            node_num=num_nodes,
        )

        graph_data = build_graph_data_for_dkn(adjacency_matrix, h_system, trajectory, measurements)
        with torch.no_grad():
            x_hat_dkn_m = kalman_process_m(graph_data).squeeze().cpu().numpy().mean(axis=1)
        x_hat_gnn_rnn_m = predict_gnn_rnn(mismatch_gnn_rnn_process, graph_data, h_system) if mismatch_gnn_rnn_process is not None else None

        x_true = trajectory[:, 0, 0].numpy()
        y_true = trajectory[:, 2, 0].numpy()

        x_cekf = x_hat_cekf_m[:, 0, 0]
        y_cekf = x_hat_cekf_m[:, 2, 0]
        trial_errors_cekf_m.append(np.sqrt((x_true - x_cekf) ** 2 + (y_true - y_cekf) ** 2).mean())

        x_dekf = x_hat_dekf_m[:, :, 0, 0].mean(axis=1)
        y_dekf = x_hat_dekf_m[:, :, 2, 0].mean(axis=1)
        trial_errors_dekf_m.append(np.sqrt((x_true - x_dekf) ** 2 + (y_true - y_dekf) ** 2).mean())

        trial_errors_dkn_m.append(np.sqrt((x_true - x_hat_dkn_m[:, 0]) ** 2 + (y_true - x_hat_dkn_m[:, 2]) ** 2).mean())
        if x_hat_gnn_rnn_m is not None:
            trial_errors_gnn_rnn_m.append(np.sqrt((x_true - x_hat_gnn_rnn_m[:, 0]) ** 2 + (y_true - x_hat_gnn_rnn_m[:, 2]) ** 2).mean())

    mismatch_errors_cekf.append(np.mean(trial_errors_cekf_m))
    mismatch_errors_dekf.append(np.mean(trial_errors_dekf_m))
    mismatch_errors_dkn.append(np.mean(trial_errors_dkn_m))
    mismatch_errors_gnn_rnn.append(np.mean(trial_errors_gnn_rnn_m) if trial_errors_gnn_rnn_m else np.nan)

mismatch_results_df = pd.DataFrame({
    "dt_mismatch": dt_mismatch_values,
    "CEKF": mismatch_errors_cekf,
    "DEKF": mismatch_errors_dekf,
    "DKN": mismatch_errors_dkn,
    "GNN-RNN": mismatch_errors_gnn_rnn,
})

mismatch_summary_path = notebook_plot_dir / f"mismatch_robustness_r={R_MISMATCH_EVAL}_trials={num_trials}.png"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["CEKF"], "o-", linewidth=2, markersize=8, label="CEKF")
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["DEKF"], "s--", linewidth=2, markersize=8, label="DEKF")
ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["DKN"], "^-.", linewidth=2, markersize=8, label="DKN")
if mismatch_results_df["GNN-RNN"].notna().any():
    ax.plot(mismatch_results_df["dt_mismatch"], mismatch_results_df["GNN-RNN"], "d:", linewidth=2, markersize=8, label="GNN-RNN")
ax.axvline(x=1.0, color="gray", linestyle=":", linewidth=1.5, label="No mismatch (ratio=1)")
ax.set_xlabel("dt mismatch ratio")
ax.set_ylabel("Average Position Error")
ax.set_title(f"dt Mismatch Robustness (r={R_MISMATCH_EVAL}, {num_trials} trials, q={process_noise_std}, T={num_time_steps})")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(mismatch_summary_path, dpi=200, bbox_inches="tight")
print(f"Saved: {mismatch_summary_path}")
plt.show()

mismatch_results_df

## 8. Node-Count Generalization

Evaluate the trained DKN and optional GNN-RNN against fresh scenarios with different node counts (the graph is rebuilt for each size). This checks how well the learned models transfer beyond their training graph size; CEKF and DEKF are included as references.

In [ ]:
TEST_NUM_NODES_VALUES = [10, 20, 30, 40]

train_num_nodes = num_nodes
NODE_COUNT_R_EVAL = measurement_noise_values[0]
NODE_COUNT_DKN_DT_RATIO = default_dkn_dt_ratio if use_dt_mismatch else None
node_count_run_name = dkn_run_name(NODE_COUNT_R_EVAL, use_dt_mismatch=use_dt_mismatch, dt_ratio=NODE_COUNT_DKN_DT_RATIO)
node_count_model_path = dkn_model_path(experiment_dir, NODE_COUNT_R_EVAL, use_dt_mismatch=use_dt_mismatch, dt_ratio=NODE_COUNT_DKN_DT_RATIO)
if not node_count_model_path.exists():
    raise FileNotFoundError(f"Missing DKN checkpoint for node-count sweep: {node_count_model_path}")

f_data_nodes = ConstantVelocityModel(time_delta)
f_model_nodes = ConstantVelocityModel(time_delta * float(NODE_COUNT_DKN_DT_RATIO)) if use_dt_mismatch else f_data_nodes

node_count_kalman_process = build_dkn_model(config_val, f_model_nodes, NODE_COUNT_R_EVAL, x_init)
node_count_kalman_process = load_state_dict_checked(node_count_kalman_process, node_count_model_path)

node_count_gnn_rnn_process = None
node_count_gnn_rnn_path = gnn_rnn_model_path(experiment_dir, NODE_COUNT_R_EVAL)
if node_count_gnn_rnn_path.exists():
    node_count_gnn_rnn_process = build_gnn_rnn_model(config_val)
    node_count_gnn_rnn_process = load_state_dict_checked(node_count_gnn_rnn_process, node_count_gnn_rnn_path)
else:
    print(f"No GNN-RNN checkpoint found at {node_count_gnn_rnn_path}; omitting GNN-RNN from node-count sweep.")

node_count_errors_cekf = []
node_count_errors_dekf = []
node_count_errors_dkn = []
node_count_errors_gnn_rnn = []

for test_num_nodes in TEST_NUM_NODES_VALUES:

    test_node_positions = generate_node_positions(test_num_nodes, seed=config_val["graph_seed"], area_size=config_val.get("area_size", 100.0))
    test_h_system = DistanceAngleObservation(test_node_positions)
    test_adjacency_matrix = create_distance_based_graph(
        test_node_positions,
        k_neighbors=config_val["k_neighbors"],
        seed=config_val["graph_seed"],
    )
    test_j_matrix = np.array(test_adjacency_matrix, dtype=float, copy=True)
    np.fill_diagonal(test_j_matrix, 1.0)
    test_r_array = NODE_COUNT_R_EVAL * np.ones(test_num_nodes)


    paired_trials = []
    for _ in range(num_trials):
        traj = generate_trajectory(f_data_nodes, x_init, num_time_steps, process_noise_std)
        meas = generate_measurements(test_h_system, traj, NODE_COUNT_R_EVAL)
        paired_trials.append((traj, meas))

    trial_errors_cekf = []
    trial_errors_dekf = []
    trial_errors_dkn = []
    trial_errors_gnn_rnn = []

    for trajectory, measurements in tqdm(paired_trials, desc=f"nodes={test_num_nodes}"):
        x_hat_cekf_n = centralized_extended_kalman_filter(
            measurements=measurements,
            f_system=f_model_nodes,
            h_system=test_h_system,
            r_array=test_r_array,
            q=process_noise_std,
            p0=p0,
            x0=x_init,
            time_steps=num_time_steps,
            node_num=test_num_nodes,
        )
        x_hat_dekf_n = diffusion_extended_kalman_filter_parallel_edge(
            measurements=measurements,
            f_system=f_model_nodes,
            h_system=test_h_system,
            r_array=test_r_array,
            q=process_noise_std,
            p0=p0,
            x0=x_init,
            j_matrix=test_j_matrix,
            time_steps=num_time_steps,
            node_num=test_num_nodes,
        )

        graph_data = build_graph_data_for_dkn(test_adjacency_matrix, test_h_system, trajectory, measurements)
        with torch.no_grad():
            x_hat_dkn_n = node_count_kalman_process(graph_data).squeeze().cpu().numpy().mean(axis=1)
        x_hat_gnn_rnn_n = predict_gnn_rnn(node_count_gnn_rnn_process, graph_data, test_h_system) if node_count_gnn_rnn_process is not None else None

        x_true = trajectory[:, 0, 0].numpy()
        y_true = trajectory[:, 2, 0].numpy()

        x_cekf = x_hat_cekf_n[:, 0, 0]
        y_cekf = x_hat_cekf_n[:, 2, 0]
        trial_errors_cekf.append(np.sqrt((x_true - x_cekf) ** 2 + (y_true - y_cekf) ** 2).mean())

        x_dekf = x_hat_dekf_n[:, :, 0, 0].mean(axis=1)
        y_dekf = x_hat_dekf_n[:, :, 2, 0].mean(axis=1)
        trial_errors_dekf.append(np.sqrt((x_true - x_dekf) ** 2 + (y_true - y_dekf) ** 2).mean())

        trial_errors_dkn.append(np.sqrt((x_true - x_hat_dkn_n[:, 0]) ** 2 + (y_true - x_hat_dkn_n[:, 2]) ** 2).mean())
        if x_hat_gnn_rnn_n is not None:
            trial_errors_gnn_rnn.append(np.sqrt((x_true - x_hat_gnn_rnn_n[:, 0]) ** 2 + (y_true - x_hat_gnn_rnn_n[:, 2]) ** 2).mean())

    node_count_errors_cekf.append(np.mean(trial_errors_cekf))
    node_count_errors_dekf.append(np.mean(trial_errors_dekf))
    node_count_errors_dkn.append(np.mean(trial_errors_dkn))
    node_count_errors_gnn_rnn.append(np.mean(trial_errors_gnn_rnn) if trial_errors_gnn_rnn else np.nan)

node_count_results_df = pd.DataFrame({
    "test_num_nodes": TEST_NUM_NODES_VALUES,
    "CEKF": node_count_errors_cekf,
    "DEKF": node_count_errors_dekf,
    "DKN": node_count_errors_dkn,
    "GNN-RNN": node_count_errors_gnn_rnn,
})

node_count_summary_path = notebook_plot_dir / f"node_count_generalization_{node_count_run_name}_trainN={train_num_nodes}_trials={num_trials}.png"
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["CEKF"], "o-", linewidth=2, markersize=8, label="CEKF")
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["DEKF"], "s--", linewidth=2, markersize=8, label="DEKF")
ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["DKN"], "^-.", linewidth=2, markersize=8, label="DKN")
if node_count_results_df["GNN-RNN"].notna().any():
    ax.plot(node_count_results_df["test_num_nodes"], node_count_results_df["GNN-RNN"], "d:", linewidth=2, markersize=8, label="GNN-RNN")
ax.axvline(x=train_num_nodes, color="gray", linestyle=":", linewidth=1.5, label=f"Train nodes ({train_num_nodes})")
ax.set_xlabel("Number of test nodes")
ax.set_ylabel("Average Position Error")
ax.set_title(f"Node-Count Generalization,fix Graph degree=5({node_count_run_name}, {num_trials} trials, T={num_time_steps})")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(node_count_summary_path, dpi=200, bbox_inches="tight")
print(f"Saved: {node_count_summary_path}")
plt.show()

node_count_results_df


## 9. Inference-Latency Comparison

Measure wall-clock inference time per trial for CEKF, DEKF, and DKN on the current device. DKN uses batched inference via `Batch.from_data_list()`; the classical filters are inherently sequential. The bar chart shows mean per-trial latency +/- one standard deviation.

In [ ]:
import time
from torch_geometric.data import Batch


LATENCY_EXP_DIR = experiment_dir
LATENCY_R = measurement_noise_values[0]
LATENCY_DT_RATIO = default_dkn_dt_ratio if use_dt_mismatch else 1.0
LATENCY_NUM_TRIALS = 200
LATENCY_BATCH_SIZE = 50
LATENCY_WARMUP = 5

lat_cfg = load_experiment_config(LATENCY_EXP_DIR)
lat_state_dim  = lat_cfg["state_dimension"]
lat_x_init     = np.array(lat_cfg["x0"], dtype=float).reshape(lat_state_dim, 1)
lat_p0         = np.eye(lat_state_dim) * lat_cfg["p0_scale"]
lat_num_nodes  = lat_cfg["num_nodes"]
lat_time_delta = lat_cfg["time_delta"]
lat_q          = lat_cfg["process_noise_std"]
lat_T          = experiment_time_steps(lat_cfg)

lat_node_positions = np.array(lat_cfg["node_positions"], dtype=float)
lat_h_system = DistanceAngleObservation(lat_node_positions)
lat_adj = create_distance_based_graph(
    lat_node_positions, k_neighbors=lat_cfg["k_neighbors"], seed=lat_cfg["graph_seed"]
)
lat_j_matrix = np.array(lat_adj, dtype=float, copy=True)
np.fill_diagonal(lat_j_matrix, 1.0)
lat_r_array  = LATENCY_R * np.ones(lat_num_nodes)
lat_f_system = ConstantVelocityModel(lat_time_delta * LATENCY_DT_RATIO)


lat_dkn_path = dkn_model_path(
    LATENCY_EXP_DIR,
    LATENCY_R,
    use_dt_mismatch=lat_cfg.get("use_dt_mismatch", False),
    dt_ratio=LATENCY_DT_RATIO,
)
if not lat_dkn_path.exists():
    raise FileNotFoundError(f"Missing DKN checkpoint: {lat_dkn_path}")

lat_dkn = build_dkn_model(lat_cfg, lat_f_system, LATENCY_R, lat_x_init)
lat_dkn = load_state_dict_checked(lat_dkn, lat_dkn_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lat_dkn = lat_dkn.to(device)
print(f"DKN device: {device}")


seed_everything(lat_cfg["seed"])
total_trials = LATENCY_NUM_TRIALS + LATENCY_WARMUP * LATENCY_BATCH_SIZE
lat_trials = []
for _ in range(total_trials):
    traj = generate_trajectory(lat_f_system, lat_x_init, lat_T, lat_q)
    meas = generate_measurements(lat_h_system, traj, LATENCY_R)
    lat_trials.append((traj, meas))


lat_data_list = [
    build_graph_data_for_dkn(lat_adj, lat_h_system, t, m) for t, m in lat_trials
]

for wi in range(LATENCY_WARMUP):
    warm_batch = Batch.from_data_list(
        lat_data_list[wi * LATENCY_BATCH_SIZE : (wi + 1) * LATENCY_BATCH_SIZE]
    ).to(device)
    with torch.no_grad():
        _ = lat_dkn(warm_batch)
sync_torch_device(device)

timing_data   = lat_data_list[LATENCY_WARMUP * LATENCY_BATCH_SIZE:]
timing_trials = lat_trials[LATENCY_WARMUP * LATENCY_BATCH_SIZE:]


cekf_times = []
for traj, meas in tqdm(timing_trials, desc="Timing CEKF"):
    t0 = time.perf_counter()
    _ = centralized_extended_kalman_filter(
        measurements=meas, f_system=lat_f_system, h_system=lat_h_system,
        r_array=lat_r_array, q=lat_q, p0=lat_p0, x0=lat_x_init,
        time_steps=lat_T, node_num=lat_num_nodes,
    )
    cekf_times.append(time.perf_counter() - t0)


dekf_times = []
for traj, meas in tqdm(timing_trials, desc="Timing DEKF"):
    t0 = time.perf_counter()
    _ = diffusion_extended_kalman_filter_parallel_edge(
        measurements=meas, f_system=lat_f_system, h_system=lat_h_system,
        r_array=lat_r_array, q=lat_q, p0=lat_p0, x0=lat_x_init,
        j_matrix=lat_j_matrix, time_steps=lat_T, node_num=lat_num_nodes,
    )
    dekf_times.append(time.perf_counter() - t0)


dkn_per_trial_ms = []
for bi in tqdm(range(0, len(timing_data), LATENCY_BATCH_SIZE), desc="Timing DKN (batch)"):
    chunk = timing_data[bi : bi + LATENCY_BATCH_SIZE]
    if not chunk:
        break
    batch = Batch.from_data_list(chunk).to(device)
    sync_torch_device(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = lat_dkn(batch)
    sync_torch_device(device)
    dkn_per_trial_ms.extend([1e3 * (time.perf_counter() - t0) / len(chunk)] * len(chunk))


results_latency = {
    "CEKF":                             np.array(cekf_times) * 1e3,
    "DEKF":                             np.array(dekf_times) * 1e3,
    f"DKN (batch={LATENCY_BATCH_SIZE})": np.array(dkn_per_trial_ms),
}

latency_df = pd.DataFrame({
    label: {"mean_ms": v.mean(), "std_ms": v.std(), "median_ms": np.median(v)}
    for label, v in results_latency.items()
}).T
print(f"\nInference latency per trial (ms) - device: {device}")
print(latency_df.round(3).to_string())


fig, ax = plt.subplots(figsize=(8, 5))
labels = list(results_latency.keys())
means  = [results_latency[l].mean() for l in labels]
stds   = [results_latency[l].std()  for l in labels]
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, edgecolor="black", linewidth=0.8)
for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        m + s + max(means) * 0.015,
        f"{m:.2f} ms",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_ylabel("Mean per-trial latency (ms)")
ax.set_title(
    f"Inference Latency Comparison\n"
    f"{LATENCY_EXP_DIR.name}, r={LATENCY_R}, dtx{LATENCY_DT_RATIO}, T={lat_T}, "
    f"nodes={lat_num_nodes}, device={device}, N={LATENCY_NUM_TRIALS} trials"
)
ax.grid(axis="y", alpha=0.35)
ax.set_ylim(0, max(means) * 1.35)

lat_plot_path = notebook_plot_dir / (
    f"inference_latency_r={LATENCY_R}_dtx{LATENCY_DT_RATIO}_batch={LATENCY_BATCH_SIZE}.png"
)
fig.savefig(lat_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved: {lat_plot_path}")
plt.show()

latency_df
